In [ ]:
import random
import pandas as pd
from pathlib import Path

random.seed(42)

# 1) 分类定义
category_meta = {
    "home":     {"is_api": False, "is_static": False, "priority": 4},
    "product":  {"is_api": False, "is_static": False, "priority": 4},
    "category": {"is_api": False, "is_static": False, "priority": 3},
    "api":      {"is_api": True,  "is_static": False, "priority": 5},
    "login":    {"is_api": False, "is_static": False, "priority": 5},
    "user":     {"is_api": False, "is_static": False, "priority": 4},
    "static":   {"is_api": False, "is_static": True,  "priority": 2},
    "admin":    {"is_api": False, "is_static": False, "priority": 3},
    "search":   {"is_api": False, "is_static": False, "priority": 3},
    "checkout": {"is_api": False, "is_static": False, "priority": 5},
    "error":    {"is_api": False, "is_static": False, "priority": 1},
    "other":    {"is_api": False, "is_static": False, "priority": 1},
}

# 2) 各分类的 URL 模板
path_templates = {
    "home": [
        "/", "/index", "/home", "/home/index", "/portal"
    ],
    "product": [
        "/product/{id}", "/product/{id}/detail", "/goods/{id}", "/sku/{id}"
    ],
    "category": [
        "/category/{slug}", "/c/{slug}", "/products/{slug}", "/list/{slug}"
    ],
    "api": [
        "/api/v1/products", "/api/v1/products/{id}",
        "/api/v1/orders/{id}", "/api/v1/cart", "/api/v1/user/{id}",
        "/api/v1/search", "/api/v1/login", "/api/v1/checkout"
    ],
    "login": [
        "/login", "/signin", "/register", "/account/login", "/auth/register"
    ],
    "user": [
        "/user", "/user/profile", "/member/{id}", "/account/orders",
        "/account/favorites", "/profile"
    ],
    "static": [
        "/static/css/app.css", "/static/js/app.js", "/static/img/logo.png",
        "/static/fonts/icon.woff2", "/assets/img/banner.jpg",
        "/img/product/{id}.jpg", "/js/lib/jquery.js"
    ],
    "admin": [
        "/admin", "/admin/dashboard", "/admin/users", "/admin/orders",
        "/admin/products", "/manage/system"
    ],
    "search": [
        "/search", "/search?q={kw}", "/s/{kw}", "/products/search/{kw}"
    ],
    "checkout": [
        "/checkout", "/checkout/cart", "/pay", "/payment/result",
        "/order/confirm", "/order/submit"
    ],
    "error": [
        "/404", "/500", "/error/404", "/error/500", "/not-found",
        "/exception", "/timeout"
    ],
    "other": [
        "/about", "/faq", "/help", "/docs", "/blog/{id}", "/contact",
        "/campaign/{id}", "/promo/{code}"
    ]
}

# 3) 生成实际 path
def build_path(category, i):
    template = random.choice(path_templates[category])
    # 生成一些真实的参数值
    replacements = {
        "{id}": str(random.randint(100, 9999)),
        "{slug}": random.choice(["electronics", "clothes", "foods", "books", "home", "beauty"]),
        "{kw}": random.choice(["phone", "shirt", "coffee", "laptop", "book", "watch"]),
        "{code}": random.choice(["VIP2024", "SALE10", "NEWUSER", "FLASH50"]),
    }
    for key, value in replacements.items():
        template = template.replace(key, value)
    return template

# 4) 生成数据
records = []
seen_paths = set()

target_rows = 800  # 800 行

while len(records) < target_rows:
    category = random.choices(
        list(category_meta.keys()),
        weights=[12, 18, 15, 10, 8, 10, 10, 6, 10, 8, 4, 8],
        k=1
    )[0]
    
    path = build_path(category, len(records))
    # 确保 path_pattern 唯一
    if path in seen_paths:
        continue
    
    seen_paths.add(path)
    meta = category_meta[category]
    records.append({
        "path_pattern": path,
        "category": category,
        "is_api": meta["is_api"],
        "is_static": meta["is_static"],
        "priority": meta["priority"],
    })

df = pd.DataFrame(records).sort_values("path_pattern").reset_index(drop=True)

# 5) 输出
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "02_url_category_dim.parquet"
df.to_parquet(output_path, index=False)

print(f"共生成 {len(df)} 条记录")
print(df.head(10))
print(f"保存路径: {output_path}")

共生成 800 条记录
         path_pattern category  is_api  is_static  priority
0                   /     home   False      False         4
1                /404    error   False      False         1
2                /500    error   False      False         1
3              /about    other   False      False         1
4  /account/favorites     user   False      False         4
5      /account/login    login   False      False         5
6     /account/orders     user   False      False         4
7              /admin    admin   False      False         3
8    /admin/dashboard    admin   False      False         3
9       /admin/orders    admin   False      False         3
保存路径: output\02_url_category_dim.parquet
